
**PALINSESTO FUTURO + ENRICHMENT AUDITEL + ENRICHMENT LLM**

OUTPUT:
1. ta_coll.whatif.output_palinsesto_rai
2. ta_coll.whatif.output_palinsesto_competitor  

FLOW:
1. Scraping palinsesto futuro (7 giorni)
2. Normalizzazione programmi futuro
3. Normalizzazione storico_programmi
4. Aggregazione feature storiche
5. Join storico -> futuro
6. Enrichment LLM
7. Output finale


## Config

### Install

In [0]:
%run ./00_utility

In [0]:
%run ../FASE1/00_utils

In [0]:
# MAGIC %pip install beautifulsoup4 unidecode --quiet

In [0]:
import os
import re
import json
import time
import requests
import warnings
import pandas as pd

from bs4 import BeautifulSoup
from unidecode import unidecode
from datetime import date, timedelta, datetime
from delta.tables import DeltaTable

from pyspark.sql import functions as F
from pyspark.sql.functions import col, avg, regexp_replace

from openai import OpenAI

warnings.filterwarnings("ignore")

### Parameters

In [0]:
# ============================================================
# TABLES
# ============================================================

AUDITEL_TABLE = "ta_coll.whatif.storico_programmi"

TABLE_RAI = "ta_coll.whatif.output_palinsesto_rai"
TABLE_COMP = "ta_coll.whatif.output_palinsesto_competitor"

# ============================================================
# SCRAPING
# ============================================================

BASE_URL = "https://www.tivu.tv/epg_ajax_sat.aspx?d={day_offset}"
GIORNI_DA_SCARICARE = 7
SCARICO_SINGOLO_GIORNO = True # se True verrà scaricato il giorno oggi + GIORNI_DA_SCARICARE-1
DELAY_REQUEST = 0.5
DELAY_DETAIL = 0.2

# ============================================================
# CANALI
# ============================================================

CANALI_RAI = ["Rai 1", "Rai 2", "Rai 3"]
CANALI_COMPETITOR = ["Rete 4", "Canale 5", "Italia 1", "LA7", "TV8", "NOVE"]
CANALI_TIVU = CANALI_RAI + CANALI_COMPETITOR

MAP_CANALE_TIVU_AUDITEL = {
    "Rai 1": "Rai 1",
    "Rai 2": "Rai 2",
    "Rai 3": "Rai 3",
    "Rete 4": "Rete 4",
    "Canale 5": "Canale 5",
    "Italia 1": "Italia 1",
    "LA7": "La7",
    "TV8": "Tv8",
    "NOVE": "Nove"
}

CANALI_TARGET = [
    "Rai 1", "Rai 2", "Rai 3",
    "Rete 4", "Canale 5", "Italia 1",
    "La7", "Tv8", "Nove"
]

# ============================================================
# LLM
# ============================================================

AZURE_OPENAI_ENDPOINT = "https://llm-whatifp-coll.openai.azure.com/openai/v1/"
AZURE_OPENAI_KEY = dbutils.secrets.get(scope="whatif-palinsesti", key="azure-openai-key")
DEPLOYMENT_NAME = "gpt-5"  
BATCH_SIZE = 10
LLM_TIMEOUT = 60

### Functions

In [0]:
def calcola_fascia(hhmm):
    try:
        h = int(hhmm[:2])
        if h < 7:
            return "night"
        if h < 21:
            return "daytime"
        return "primetime"
    except:
        return "night"


def calcola_durata(start, end):
    try:
        s = pd.to_datetime(start, format="%H:%M")
        e = pd.to_datetime(end, format="%H:%M")
        if e < s:
            e += pd.Timedelta(days=1)
        return int((e - s).total_seconds() / 60)
    except:
        return None


ETA_COLS = {
    "15_24": "1st_Screen_LiveVOSDAL_Adulti_15_24",
    "25_34": "1st_Screen_LiveVOSDAL_Adulti_25_34",
    "35_44": "1st_Screen_LiveVOSDAL_Adulti_35_44",
    "45_54": "1st_Screen_LiveVOSDAL_Adulti_45_54",
    "55_64": "1st_Screen_LiveVOSDAL_Adulti_55_64",
    "65_69": "1st_Screen_LiveVOSDAL_Adulti_65_69",
    "70_74": "1st_Screen_LiveVOSDAL_Adulti_70_74",
    "75_plus": "1st_Screen_LiveVOSDAL_Adulti_75plus"
}

In [0]:
def fetch_epg(day_offset):
    r = requests.get(
        BASE_URL.format(day_offset=day_offset),
        timeout=15
    )
    r.raise_for_status()
    return r.text

def parse_epg(html, canali_target, data_riferimento):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for ch_div in soup.find_all("div", class_="q"):
        first_link = ch_div.find("a", attrs={"data-channel": True})
        if not first_link:
            continue
        channel = first_link.get("data-channel", "").strip()
        if channel not in canali_target:
            continue
        prog_divs = ch_div.find_all(
            "div",
            class_=re.compile(r"^p")
        )
        programmi = []

        # ====================================================
        # ESTRAZIONE PROGRAMMI
        # ====================================================
        for prog_div in prog_divs:
            text = prog_div.get_text(" ", strip=True)
            # prende tutti gli orari presenti
            times = re.findall(r"\d{2}:\d{2}", text)
            if not times:
                continue
            orario_inizio = times[0]
            # titolo = testo senza orario
            titolo = re.sub(r"\d{2}:\d{2}", "", text)
            titolo = re.sub(r"\s+", " ", titolo).strip()
            programmi.append({
                "titolo": titolo,
                "orario_inizio": orario_inizio
            })

        # ====================================================
        # CALCOLO ORARIO_FINE
        # ====================================================

        for i in range(len(programmi)):
            current = programmi[i]
            orario_fine = None
            # fine = inizio programma successivo
            if i < len(programmi) - 1:
                orario_fine = programmi[i + 1]["orario_inizio"]
            else:
                # ultimo programma giornata
                orario_fine = "05:59"
            rows.append({
                "Data": data_riferimento,
                "Canale": channel,
                "Programma": current["titolo"],
                "orario_inizio": current["orario_inizio"],
                "orario_fine": orario_fine
            })

    return rows

In [0]:
def is_night(val):
    try:
        return int(str(val).split(":")[0]) < 6
    except:
        return False
    
client = OpenAI(
    api_key=AZURE_OPENAI_KEY,
    base_url=AZURE_OPENAI_ENDPOINT
)

def call_llm(messages, temperature=1):
    response = client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=messages,
        temperature=temperature    
        )
    return response.choices[0].message.content

def build_prompt(ctx):
    return f"""
Sei un analista TV.

DATI AUDITEL:
{ctx}

Per ogni programma restituisci JSON array:

- gid (obbligatorio)
- evento_forte (true/false)

Regole:
- non inventare dati
- match esatto titoli
"""

def enrich_with_llm(df, context):
    system_prompt = build_prompt(context)
    programs = df.to_dict("records")

    results = {}

    n_batches = (len(programs) + BATCH_SIZE - 1) // BATCH_SIZE

    for b_start in range(0, len(programs), BATCH_SIZE):

        batch = programs[b_start:b_start + BATCH_SIZE]

        lines = []

        for i, p in enumerate(batch):
            gid = b_start + i

            lines.append(
                f"[gid={gid}] Canale: {p['Canale']} | "
                f"Orario: {p['orario_inizio']}-{p['orario_fine']} | "
                f"Programma: {p['Programma']}"
            )

        user_prompt = "Analizza programmi TV:\n\n" + "\n".join(lines)

        print(f"Batch {b_start//BATCH_SIZE + 1}/{n_batches}")

        raw = call_llm([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ])

        try:
            data = json.loads(raw.replace("```json", "").replace("```", ""))
        except Exception as e:
            print("JSON ERROR:", e)
            continue

        for item in data:
            gid = item.get("gid")
            if gid is not None:
                results[int(gid)] = item

    return results

def load_auditel_context(spark_session, canali_filter):
    from pyspark.sql.functions import col, avg

    df = spark_session.table(AUDITEL_TABLE)

    summary = (
        df.filter(col("Canale").isin(canali_filter))
          .groupBy("Programma", "Canale")
          .agg(
              avg("Share").alias("Share_medio"),
              avg("Total_Audience").alias("Audience_medio")
          )
          .orderBy(col("Audience_medio").desc())
          .limit(300)
    )

    return "\n".join(summary.toJSON().collect())

## Scraping Palinsesto Futuro

In [0]:
all_rows = []
oggi = date.today()

if SCARICO_SINGOLO_GIORNO:
    offset_list = [GIORNI_DA_SCARICARE-1]
else:
    offset_list = range(GIORNI_DA_SCARICARE)

for offset in offset_list:
    giorno = oggi + timedelta(days=offset)
    try:
        html = fetch_epg(offset)
        rows = parse_epg(
            html,
            CANALI_TIVU,
            giorno
        )
        all_rows.extend(rows)
        print(f"{giorno}: {len(rows)} righe")

    except Exception as e:
        print(f"ERRORE {giorno}: {e}")
    time.sleep(DELAY_REQUEST)

future_df = pd.DataFrame(all_rows)
# Elimina programmi della fascia notturna 01:00 - 05:59
future_df["ora"] = future_df["orario_inizio"].str[:2].astype(int)

future_df = future_df[
    ~future_df["ora"].between(1, 5)
].copy()

print(f"Totale righe dopo filtro 01-06: {len(future_df):,}")

In [0]:
future_df.display()

## Normalizzazione Palinsesto Futuro

In [0]:
future_df["programma_norm"] = future_df["Programma"].apply(normalize_title)
future_df["programma_norm"] = [
    apply_manual_mapping(c, p)
    for c, p in zip(future_df["Canale"], future_df["programma_norm"])
]

future_df["Programma"] = future_df["Programma"].str.replace(" -", "", regex=False)

future_df["canale_norm"] = (
    future_df["Canale"]
    .map(MAP_CANALE_TIVU_AUDITEL)
    .fillna(future_df["Canale"])
)


In [0]:

future_df["fascia_oraria"] = (
    future_df["orario_inizio"]
    .apply(calcola_fascia)
)

future_df["durata_minuti"] = future_df.apply(
    lambda x: calcola_durata(
        x["orario_inizio"],
        x["orario_fine"]
    ),
    axis=1
)

future_df["giorno_settimana"] = pd.to_datetime(
    future_df["Data"]
).dt.weekday

future_df["ora"] = (
    future_df["orario_inizio"]
    .str[:2]
    .astype(int)
)

In [0]:
future_df.display()

## Loading Palinsesto Storico

In [0]:
storico_df = read_df_programmi(
    table_name=AUDITEL_TABLE
)

## Normalizzazione Palinsesto Storico

In [0]:
storico_df = storico_df[storico_df["Canale"].isin(CANALI_TARGET)].copy()

In [0]:
%skip
storico_df["programma_norm"] = (
    storico_df["Programma"]
    .apply(normalize_title)
)
storico_df["programma_norm"] = [
    apply_manual_mapping(c, p)
    for c, p in zip(storico_df["Canale"], storico_df["programma_norm"])
]


In [0]:
storico_df["canale_norm"] = (
    storico_df["Canale"]
)

storico_df["giorno_settimana"] = pd.to_datetime(
    storico_df["Data"]
).dt.weekday

storico_df["ora"] = (
    storico_df["ORA_INIZIO_TRX"] // 3600
).astype("Int64")

In [0]:
storico_df.display()

## Arricchimento da Storico

### Calcolo Feature Storiche

In [0]:
# ============================================================
# SHARE STORICO
# ============================================================
agg_share = (
    storico_df
    .groupby([
        "programma_norm",
        "canale_norm",
        "ora",
        "giorno_settimana"    
        ])
    .agg(
        share_storico=("Share", "mean")
    )
    .reset_index()
)

# ============================================================
# GENERE
# ============================================================
agg_genere = (
    storico_df
    .groupby([
        "programma_norm",
        "canale_norm",
        "ora",
        "giorno_settimana" 
    ])
    ["DES_GENERE_ESTESA_INT"]
    .agg(lambda x: x.value_counts().index[0] if len(x.dropna()) > 0 else None)
    .reset_index(name="genere_predominante")
)

# ============================================================
# TARGET GENERE
# ============================================================
def compute_gender_target(group):

    uomini = group["1st_Screen_LiveVOSDAL_Uomini"].median()
    donne = group["1st_Screen_LiveVOSDAL_Donne"].median()

    if pd.isna(uomini) or pd.isna(donne):
        return None

    if abs(uomini - donne) < 5:
        return "Misto"

    return "Uomini" if uomini > donne else "Donne"


gender_rows = []

for keys, group in storico_df.groupby([
    "programma_norm",
    "canale_norm",
    "ora",
    "giorno_settimana" 
]):

    gender_rows.append({
        "programma_norm": keys[0],
        "canale_norm": keys[1],
        "ora": keys[2],
        "giorno_settimana": keys[3],
        "target_genere": compute_gender_target(group)
    })

gender_df = pd.DataFrame(gender_rows)

# ============================================================
# TARGET ETA
# ============================================================
eta_rows = []

for keys, group in storico_df.groupby([
    "programma_norm",
    "canale_norm",
    "ora",
    "giorno_settimana" 
]):

    medians = {}

    for label, colname in ETA_COLS.items():

        medians[label] = group[colname].median()

    eta_top = max(medians, key=medians.get)

    eta_rows.append({
        "programma_norm": keys[0],
        "canale_norm": keys[1],
        "ora": keys[2],
        "giorno_settimana": keys[3],
        "target_eta": eta_top
    })

eta_df = pd.DataFrame(eta_rows)

### Aggancio Feature Storiche

In [0]:
future_df = future_df.merge(
    agg_share,
    on=["programma_norm", "canale_norm", "ora", "giorno_settimana"],
    how="left"
)

future_df = future_df.merge(
    agg_genere,
    on=["programma_norm", "canale_norm", "ora", "giorno_settimana"],
    how="left"
)

In [0]:
future_df = future_df.merge(
    gender_df,
    on=["programma_norm", "canale_norm", "ora", "giorno_settimana"],
    how="left"
)

future_df = future_df.merge(
    eta_df,
    on=["programma_norm", "canale_norm", "ora", "giorno_settimana"],
    how="left"
)

## Divisione Rai e Competitor

In [0]:

df_rai_future = future_df[
    future_df["Canale"].isin(CANALI_RAI)
].copy()

df_comp_future = future_df[
    future_df["Canale"].isin(CANALI_COMPETITOR)
].copy()


print(f"RAI: {len(df_rai_future):,}")
print(f"COMP: {len(df_comp_future):,}")


In [0]:
if len(df_rai_future) == 0 or len(df_comp_future)==0:
    dbutils.notebook.exit("Dati mancanti per il palinsesto futuro")

In [0]:
df_rai_future = df_rai_future[[
    "Data",
    "canale_norm",
    "Programma",
    "programma_norm",
    "orario_inizio",
    "orario_fine",
    "fascia_oraria",
    "durata_minuti",
    "giorno_settimana",
    "ora",
    "share_storico",
    "genere_predominante",
    "target_genere",
    "target_eta"
]]

df_comp_future = df_comp_future[[
    "Data",
    "canale_norm",
    "Programma",
    "programma_norm",
    "orario_inizio",
    "orario_fine",
    "fascia_oraria",
    "durata_minuti",
    "giorno_settimana",
    "ora",
    "share_storico",
    "genere_predominante",
    "target_genere",
    "target_eta"
]]

In [0]:
df_rai_future = df_rai_future.rename(columns={
#     "programma_norm": "Programma",
     "canale_norm": "Canale"
 })

df_comp_future = df_comp_future.rename(columns={
#     "programma_norm": "Programma",
     "canale_norm": "Canale"
 })

In [0]:
df_rai_future.display()

In [0]:
df_comp_future.display()

## Arricchimento da LLM

In [0]:
ctx_rai = load_auditel_context(spark, ["Rai 1", "Rai 2", "Rai 3"])
ctx_comp = load_auditel_context(spark, CANALI_COMPETITOR)

df_rai = df_rai_future.copy().reset_index(drop=True)
df_comp = df_comp_future.copy().reset_index(drop=True)

print("RAI:", len(df_rai))
print("COMP:", len(df_comp))

In [0]:

import json
import pandas as pd

def is_night(val):
    try:
        hour = int(str(val).split(":")[0])
        return hour < 6
    except Exception:
        return False


def call_llm(messages):
    try:
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=messages,
            response_format={"type": "json_object"}  # se supportato dal tuo deployment
        )
        return response.choices[0].message.content

    except Exception as e:
        print("LLM ERROR:", e)
        return '{"items":[]}'


def build_prompt(ctx):
    return f"""
Sei un analista TV.

DATI AUDITEL DI RIFERIMENTO:
{ctx}

Restituisci SOLO JSON valido nel formato:

{{
  "items": [
    {{
      "gid": 0,
      "evento_forte": true
    }}
  ]
}}

Regole:
- usa solo i programmi forniti dall'utente
- non inventare dati
- non modificare i titoli
- se non sei sicuro, usa evento_forte=false
"""


def safe_json_load(raw):
    try:
        raw = raw.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        data = json.loads(raw)

        if isinstance(data, list):
            return data

        if isinstance(data, dict):
            return data.get("items", [])

        return []

    except Exception as e:
        print("JSON ERROR:", e)
        print("RAW:", raw[:1000])
        return []


def enrich_with_llm(df, context, batch_size=BATCH_SIZE):
    system_prompt = build_prompt(context)

    programs = df.reset_index(drop=True).to_dict("records")
    results = {}

    n_batches = (len(programs) + batch_size - 1) // batch_size

    for b_start in range(0, len(programs), batch_size):
        batch = programs[b_start:b_start + batch_size]

        lines = []
        for i, p in enumerate(batch):
            gid = b_start + i
            lines.append(
                f"[gid={gid}] "
                f"Canale: {p.get('Canale', '')} | "
                f"Orario: {p.get('orario_inizio', '')}-{p.get('orario_fine', '')} | "
                f"Programma: {p.get('Programma', '')}"
            )

        user_prompt = "Analizza questi programmi TV:\n\n" + "\n".join(lines)

        print(f"Batch {b_start // batch_size + 1}/{n_batches}")

        raw = call_llm([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ])

        data = safe_json_load(raw)

        for item in data:
            gid = item.get("gid")

            if gid is None:
                continue

            try:
                gid = int(gid)
            except Exception:
                continue

            results[gid] = {
                "evento_forte": bool(item.get("evento_forte", False))
            }

    return results


def make_key(r):
    return (
        str(r.get("Canale", "")).strip().lower(),
        str(r.get("Programma", "")).strip().lower()
    )


def build_enrichment_mapping(df, llm_results):
    mapping = {}

    for gid, row in enumerate(df.reset_index(drop=True).to_dict("records")):
        r = llm_results.get(gid, {})
        mapping[make_key(row)] = {
            "evento_forte": bool(r.get("evento_forte", False))
        }

    return mapping


def apply_enrichment(row, mapping):
    feats = dict(mapping.get(make_key(row), {"evento_forte": False}))

    if is_night(row.get("orario_inizio")):
        feats["evento_forte"] = False

    return pd.Series(feats)


def enrich_dataframe(df, ctx, label):
    llm_results = enrich_with_llm(df, ctx)
    mapping = build_enrichment_mapping(df, llm_results)

    enriched = df.copy()
    enriched[["evento_forte"]] = enriched.apply(
        lambda r: apply_enrichment(r, mapping),
        axis=1
    )

    print(f"\n{label}:")
    print(enriched["evento_forte"].value_counts(dropna=False))

    return enriched, mapping, llm_results

In [0]:
df_rai, enrichment_rai, llm_rai = enrich_dataframe(df_rai, ctx_rai, "RAI")

In [0]:
df_comp, enrichment_comp, llm_comp = enrich_dataframe(df_comp, ctx_comp, "COMP")

In [0]:
df_comp.display()

## Salvataggio Tabelle


In [0]:
df_rai = df_rai.drop_duplicates(['Data','Canale','orario_inizio','programma_norm'])
df_rai = spark.createDataFrame(df_rai)
df_rai = df_rai.filter(F.col('orario_fine') > F.col('orario_inizio'))
df_rai = df_rai.withColumn('ID', F.concat(F.col('Canale'), F.lit('_'), F.col('Data'), F.lit('_'), F.col('programma_norm'), F.lit('_'), F.col('orario_inizio')))

# if table doesn't exist in UC --> create it
if not spark.catalog.tableExists(TABLE_RAI):
    df_rai.write.format("delta").saveAsTable(TABLE_RAI)
else:
    # otherwise do an upsert
    delta_table_rai = DeltaTable.forName(spark, TABLE_RAI)

    (
        delta_table_rai.alias("target")
        .merge(
            df_rai.alias("source"),
            "target.Data = source.Data AND target.Canale = source.Canale AND target.orario_inizio = source.orario_inizio AND target.programma_norm = source.programma_norm"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
df_comp = df_comp.drop_duplicates(['Data','Canale','orario_inizio','programma_norm'])
df_comp = spark.createDataFrame(df_comp)
df_comp = df_comp.filter(F.col('orario_fine') > F.col('orario_inizio'))
df_comp = df_comp.withColumn('ID', F.concat(F.col('Canale'), F.lit('_'), F.col('Data'), F.lit('_'), F.col('programma_norm'), F.lit('_'), F.col('orario_inizio')))

# if table doesn't exist in UC --> create it
if not spark.catalog.tableExists(TABLE_COMP):
    df_comp.write.format("delta").saveAsTable(TABLE_COMP)
else:
    # otherwise do an upsert
    delta_table_comp = DeltaTable.forName(spark, TABLE_COMP)

    (
        delta_table_comp.alias("target")
        .merge(
            df_comp.alias("source"),
            "target.Data = source.Data AND target.Canale = source.Canale AND target.orario_inizio = source.orario_inizio AND target.programma_norm = source.programma_norm"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

## Creazione Vista

In [0]:
%sql
CREATE VIEW IF NOT EXISTS ta_coll.whatif.vw_output_palinsesto_futuro AS
-- CREATE OR REPLACE VIEW ta_coll.whatif.vw_output_palinsesto_futuro AS
-- Palinsesto unificato: unisce in una sola riga sia i programmi con righe consecutive (split editoriali, gap ≈0) sia quelli interrotti da programmi più corti (spot, meteo...).
-- Palinsesto completo
WITH palinstesto AS (
    SELECT
        Canale, Data, Programma, programma_norm,
        orario_inizio, orario_fine, share_storico, evento_forte
    FROM ta_coll.whatif.output_palinsesto_rai
    UNION ALL
    SELECT
        Canale, Data, Programma, programma_norm,
        orario_inizio, orario_fine, share_storico, evento_forte
    FROM ta_coll.whatif.output_palinsesto_competitor
),
-- Calcolo dell'orario in secondi
base AS (
    SELECT
        Canale, Data, Programma, programma_norm, 
        orario_inizio, orario_fine, share_storico,
        (CAST(SPLIT(orario_inizio, ':')[0] AS INT) * 3600 + CAST(SPLIT(orario_inizio, ':')[1] AS INT) * 60) AS inizio_sec,
        (CAST(SPLIT(orario_fine, ':')[0] AS INT) * 3600 + CAST(SPLIT(orario_fine, ':')[1] AS INT) * 60) AS fine_sec_raw,
        evento_forte
    FROM palinstesto
),
-- Orario finale corretto per i programmi che iniziano prima di mezzanotte ma finiscono dopo
midnight_adjusted AS (
    SELECT
        *,
        CASE
            WHEN fine_sec_raw < inizio_sec
            THEN fine_sec_raw + 86400
            ELSE fine_sec_raw
        END AS fine_sec
    FROM base
),
-- Flag per i programmi consecutivi da aggregare e i programmi di interruzione
flagged AS (
    SELECT
        Canale, Data, Programma, programma_norm, 
        orario_inizio, 
        orario_fine,
        inizio_sec,
        fine_sec,
        share_storico,
        evento_forte,
        -- Flag programmi consecutivi da aggregare
        CASE
            WHEN inizio_sec - LAG(fine_sec) OVER w1 <= 900 -- se il gap tra due consecutivi e' <=15 min, viene considerato lo stesso programma
            THEN 0 ELSE 1
        END AS is_new_group,
        -- Flag programma interrutore
        CASE
            WHEN LAG(programma_norm) OVER w2 != programma_norm -- diverso dal precedente
            AND LAG(programma_norm) OVER w2 = LEAD(programma_norm) OVER w2 -- prima e dopo c'e' lo stesso programma
            AND inizio_sec - LAG(fine_sec) OVER w2 <= 300 -- inizia entro i 5 minuti dal precedente
            AND (LEAD(inizio_sec) OVER w2) - fine_sec <= 300 -- finisce entro i 5 minuti dal successivo
            AND (fine_sec - inizio_sec) <= 900 -- la riga stessa dura massimo 15 minuti
            THEN 1 
            ELSE 0
        END AS is_interruptor
    FROM midnight_adjusted
    WINDOW 
        w1 AS (PARTITION BY Data, Canale, programma_norm ORDER BY inizio_sec),
        w2 AS (PARTITION BY Data, Canale ORDER BY inizio_sec)
),
-- Flag per separare i gruppi di programmi
grouped AS (
    SELECT
        Canale, Data, Programma, programma_norm,
        orario_inizio, orario_fine,
        inizio_sec, fine_sec, share_storico,
        evento_forte,
        SUM(is_new_group) OVER (
            PARTITION BY Data, Canale, programma_norm
            ORDER BY inizio_sec
        ) AS grp
    FROM flagged
)
-- Raggruppamento e creazione ID
SELECT
    CONCAT(Canale, '_', CAST(Data AS STRING), '_', programma_norm, '_', MIN_BY(orario_inizio, inizio_sec)) AS ID,
    Canale, Data,
    MIN_BY(Programma, inizio_sec) AS Programma,
    MIN_BY(orario_inizio, inizio_sec) AS orario_inizio,
    MAX_BY(orario_fine, fine_sec) AS orario_fine,
    AVG(share_storico) AS share_storico,
    MAX(evento_forte) AS evento_forte
FROM grouped
GROUP BY Data, Canale, programma_norm, grp
ORDER BY orario_inizio
;

In [0]:
%sql
SELECT * FROM ta_coll.whatif.vw_output_palinsesto_futuro
ORDER BY Data DESC